In [ ]:
"""
FakeAVCeleb Deepfake Detection - Model Architectures
=====================================================

Karşılaştırmalı backbone denemesi için üç model:
  1. EfficientNet-B4 + BiLSTM  (mevcut)
  2. ResNet-50       + BiLSTM
  3. ResNeXt-50      + BiLSTM

Aynı BiLSTM kafası, sadece backbone değişiyor → adil karşılaştırma.
"""

import torch
import torch.nn as nn
from torchvision.models import (
    efficientnet_b4, EfficientNet_B4_Weights,
    resnet50,        ResNet50_Weights,
    resnext50_32x4d, ResNeXt50_32X4D_Weights,
)
from einops import rearrange

# ─── CONFIG ───────────────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 16
IMAGE_SIZE      = 224
HIDDEN_DIM      = 256
DROPOUT         = 0.5
# ──────────────────────────────────────────────────────────────────────────────


# ═════════════════════════════════════════════════════════════════════════════
#  ORTAK BiLSTM KAFA  —  tüm modeller bunu paylaşıyor
# ═════════════════════════════════════════════════════════════════════════════

class BiLSTMHead(nn.Module):
    """
    CNN feature vektörlerini (B, T, cnn_dim) alır,
    BiLSTM ile zamansal modelleme yapar,
    (B, 1) logit döndürür.
    """

    def __init__(self, cnn_dim: int, hidden_dim: int = HIDDEN_DIM,
                 dropout: float = DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=cnn_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 1),
        )

    def forward(self, feats):
        # feats: (B, T, cnn_dim)
        lstm_out, _ = self.lstm(feats)   # (B, T, hidden*2)
        out = lstm_out[:, -1, :]         # son timestep → (B, hidden*2)
        return self.classifier(out)      # (B, 1)


# ═════════════════════════════════════════════════════════════════════════════
#  GENEL SARICI  —  herhangi bir backbone + BiLSTMHead
# ═════════════════════════════════════════════════════════════════════════════

class CNNBiLSTM(nn.Module):
    """
    Herhangi bir CNN backbone + BiLSTM.
    Input:  (B, T, C, H, W)
    Output: (B, 1)
    """

    def __init__(self, backbone: nn.Module, cnn_dim: int,
                 hidden_dim: int = HIDDEN_DIM, dropout: float = DROPOUT):
        super().__init__()
        self.cnn  = backbone
        self.head = BiLSTMHead(cnn_dim, hidden_dim, dropout)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x     = rearrange(x, 'b t c h w -> (b t) c h w')
        feats = self.cnn(x).flatten(1)                        # (B*T, cnn_dim)
        feats = rearrange(feats, '(b t) d -> b t d', b=B, t=T)
        return self.head(feats)                               # (B, 1)


# ─── Backbone fabrikaları ─────────────────────────────────────────────────────

def _efficientnet_b4_backbone(freeze_ratio: float = 0.8):
    """EfficientNet-B4 → 1792 boyutlu özellik vektörü."""
    base = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
    backbone = nn.Sequential(*list(base.children())[:-1])   # AdaptiveAvgPool dahil

    all_params = list(backbone.parameters())
    freeze_until = int(len(all_params) * freeze_ratio)
    for i, p in enumerate(all_params):
        p.requires_grad = (i >= freeze_until)

    return backbone, 1792


def _resnet50_backbone(freeze_ratio: float = 0.8):
    """ResNet-50 → 2048 boyutlu özellik vektörü."""
    base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    # fc katmanını at, geri kalanı tut
    backbone = nn.Sequential(*list(base.children())[:-1])   # (B, 2048, 1, 1)

    all_params = list(backbone.parameters())
    freeze_until = int(len(all_params) * freeze_ratio)
    for i, p in enumerate(all_params):
        p.requires_grad = (i >= freeze_until)

    return backbone, 2048


def _resnext50_backbone(freeze_ratio: float = 0.8):
    """ResNeXt-50-32x4d → 2048 boyutlu özellik vektörü."""
    base = resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V2)
    backbone = nn.Sequential(*list(base.children())[:-1])   # (B, 2048, 1, 1)

    all_params = list(backbone.parameters())
    freeze_until = int(len(all_params) * freeze_ratio)
    for i, p in enumerate(all_params):
        p.requires_grad = (i >= freeze_until)

    return backbone, 2048


# ─── Ana fabrika fonksiyonu ───────────────────────────────────────────────────

_BACKBONE_REGISTRY = {
    "efficientnet": (_efficientnet_b4_backbone, "EfficientNet-B4 + BiLSTM"),
    "resnet":       (_resnet50_backbone,        "ResNet-50      + BiLSTM"),
    "resnext":      (_resnext50_backbone,       "ResNeXt-50     + BiLSTM"),
}


def build_model(model_type: str = "efficientnet",
                hidden_dim: int = HIDDEN_DIM,
                dropout: float  = DROPOUT,
                freeze_ratio: float = 0.8,
                **kwargs) -> nn.Module:
    """
    model_type: "efficientnet" | "resnet" | "resnext"

    Örnek:
        model = build_model("resnet")
        model = build_model("resnext")
        model = build_model("efficientnet")
    """
    if model_type not in _BACKBONE_REGISTRY:
        raise ValueError(
            f"Bilinmeyen model tipi: '{model_type}'. "
            f"Seçenekler: {list(_BACKBONE_REGISTRY.keys())}"
        )

    factory_fn, label = _BACKBONE_REGISTRY[model_type]
    backbone, cnn_dim = factory_fn(freeze_ratio)
    model = CNNBiLSTM(backbone, cnn_dim, hidden_dim, dropout)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model   : {label}")
    print(f"CNN dim : {cnn_dim}")
    print(f"Params  : {total:,} total | {trainable:,} trainable")
    return model


# ─── Hızlı test ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    dummy = torch.randn(2, 16, 3, 224, 224)
    for mtype in ["efficientnet", "resnet", "resnext"]:
        print(f"\n{'─'*50}")
        m = build_model(mtype)
        out = m(dummy)
        print(f"Çıktı şekli: {out.shape}")   # (2, 1)